In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/tupy-e/binary_test.csv
/kaggle/input/tupy-e/binary_train.csv
/kaggle/input/tupy-e-bert/config.json
/kaggle/input/tupy-e-bert/training_args.bin
/kaggle/input/tupy-e-bert/tokenizer_config.json
/kaggle/input/tupy-e-bert/model.safetensors
/kaggle/input/tupy-e-bert/special_tokens_map.json
/kaggle/input/tupy-e-bert/vocab.txt


In [2]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU is enabled. Using {torch.cuda.get_device_name(0)}.")
else:
    print("GPU is not enabled. Using CPU.")

GPU is enabled. Using Tesla T4.


In [3]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import torch.nn as nn

# Load the trained model and tokenizer
tokenizer = BertTokenizer.from_pretrained("/kaggle/input/tupy-e-bert")
model = BertForSequenceClassification.from_pretrained("/kaggle/input/tupy-e-bert")

2025-05-02 18:49:48.300049: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746211788.477450      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746211788.528874      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
bert_model = model.bert  # Access the BERT model

In [5]:
class CustomBertLSTMModel(nn.Module):
    def __init__(self, bert_model, hidden_size=768, lstm_hidden_size=128, num_labels=2):
        super(CustomBertLSTMModel, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.1)
        self.lstm = nn.LSTM(hidden_size, lstm_hidden_size, batch_first=True)
        self.classifier = nn.Linear(lstm_hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        # Get BERT outputs
        outputs = self.bert(input_ids=input_ids,
                           attention_mask=attention_mask,
                           token_type_ids=token_type_ids)
        last_hidden_state = outputs[0]  # Shape: (batch_size, seq_len, hidden_size)
        last_hidden_state = self.dropout(last_hidden_state)
        
        # Pass through LSTM
        lstm_output, (hn, cn) = self.lstm(last_hidden_state)  # hn: (num_layers, batch_size, lstm_hidden_size)
        lstm_out = hn[-1]  # Take the last hidden state of LSTM
        
        # Classifier
        logits = self.classifier(lstm_out)
        return logits
custom_model = CustomBertLSTMModel(bert_model, lstm_hidden_size=128, num_labels=2)

In [6]:
from datasets import load_dataset,Dataset

In [7]:
# Load the training data
train_df = pd.read_csv('/kaggle/input/tupy-e/binary_train.csv')
train_df = train_df[['text', 'hate','aggressive']].dropna()

# Load the test data
test_df = pd.read_csv('/kaggle/input/tupy-e/binary_test.csv')
test_df = test_df[['text', 'hate','aggressive']].dropna()


In [8]:
from sklearn.model_selection import train_test_split
import re
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()  # Lowercase
    text = re.sub(r'@\w+', '', text)  # Remove @mentions
    text = re.sub(r'http\S+|www.\S+', '', text)  # Remove URLs
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    return text.strip()
train_df['text'] = train_df['text'].apply(preprocess_text)
test_df['text'] = test_df['text'].apply(preprocess_text)
label = np.zeros((len(train_df['text']), 2))
train_labels = train_df['hate'].tolist()
for i in range(len(train_df)):
    if train_df['hate'][i] == 0:
        label[i] = [1, 0]
    else:
        label[i] = [0, 1]
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_labels,
    test_size=0.2,
    random_state=42
)
train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_labels}) # Changed 'hate' to 'labels'
val_dataset = Dataset.from_dict({'text': val_texts, 'labels': val_labels}) # Changed 'hate' to 'labels'

# For testing
test_labels = [[1, 0] if label == 0 else [0, 1] for label in test_df['hate'].tolist()]
test_dataset = Dataset.from_dict({'text': test_df['text'].tolist(), 'labels': test_labels}) # Changed 'hate' to 'labels'

# Tokenize all
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/27947 [00:00<?, ? examples/s]

Map:   0%|          | 0/6987 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

In [9]:
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm 

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
custom_model.to(device)

# Freeze BERT layers (optional, to train only LSTM and classifier)
for param in custom_model.bert.parameters():
    param.requires_grad = False

# Optimizer
optimizer = optim.AdamW([
    {"params": custom_model.lstm.parameters(), "lr": 1e-3},
    {"params": custom_model.classifier.parameters(), "lr": 1e-3}
])

# Loss function
criterion = nn.CrossEntropyLoss()

# DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16)
test_dataloader = DataLoader(test_dataset, batch_size=16)

# Training loop
num_epochs = 10
custom_model.train()

for epoch in tqdm(range(num_epochs)):
    total_loss = 0
    for batch in train_dataloader:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        optimizer.zero_grad()  # Clear previous gradients
        logits = custom_model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        # Backward pass
        loss.backward()  # Compute gradients
        optimizer.step()  # Update parameters

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")

    # Validation
    custom_model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = custom_model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average='binary')
    print(f"Validation Accuracy: {accuracy:.4f}, F1 Score: {f1:.4f}")
    custom_model.train()


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1/10, Average Loss: 0.2063


 10%|█         | 1/10 [05:17<47:36, 317.42s/it]

Validation Accuracy: 0.9044, F1 Score: 0.5229
Epoch 2/10, Average Loss: 0.2019


 20%|██        | 2/10 [10:59<44:17, 332.21s/it]

...

Validation Accuracy: 0.8879, F1 Score: 0.5569
Epoch 9/10, Average Loss: 0.1878


 90%|█████████ | 9/10 [51:08<05:43, 343.33s/it]

Validation Accuracy: 0.9058, F1 Score: 0.5637
Epoch 10/10, Average Loss: 0.1875


100%|██████████| 10/10 [56:50<00:00, 341.06s/it]

Validation Accuracy: 0.9021, F1 Score: 0.5470


In [10]:

# Save the model
custom_model.bert.save_pretrained("/kaggle/working/my-trained-bertimbau-lstm-model")
tokenizer.save_pretrained("/kaggle/working/my-trained-bertimbau-lstm-model")
torch.save(custom_model.state_dict(), "/kaggle/working/my-trained-bertimbau-lstm-model/pytorch_model.bin")